# ZEREOS AI Shorts — Free temporary GPU worker

This notebook starts the real MuseTalk worker for the GitHub Pages studio. In Colab, choose **Runtime → Change runtime type → T4 GPU** (if available), then run the cells from top to bottom. Free GPU access and session duration are controlled by Google and are not guaranteed. Large model downloads may take a while. Use only media you own or have permission to use.

In [ ]:
import torch, os, subprocess, sys
if not torch.cuda.is_available():
    raise RuntimeError('GPU غير متاحة. اختر Runtime > Change runtime type > T4 GPU ثم أعد التشغيل.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
WORK='/content/Zereos-AI-Shorts'
if not os.path.isdir(WORK):
    subprocess.run(['git','clone','--depth','1','https://github.com/arkanws513-spec/Zereos-AI-Shorts.git',WORK],check=True)
os.chdir(WORK)
print('Repository ready:', WORK)

In [ ]:
import os, subprocess
ENGINE='/content/MuseTalk'
if not os.path.isdir(ENGINE):
    subprocess.run(['git','clone','--depth','1','https://github.com/TMElyralab/MuseTalk.git',ENGINE],check=True)
os.chdir(ENGINE)
# MuseTalk's pinned requirements may adjust packages already present in Colab.
subprocess.run(['pip','install','-r','requirements.txt'],check=True)
subprocess.run(['pip','install','fastapi==0.115.12','uvicorn[standard]==0.34.0','python-multipart==0.0.20','PyYAML==6.0.2'],check=True)
print('MuseTalk dependencies installed. If pip reports conflicts, restart the runtime and rerun from the first cell.')

In [ ]:
import os, subprocess
os.chdir('/content/MuseTalk')
subprocess.run(['bash','download_weights.sh'],check=True)
print('Weight download command completed. Next cell checks every required file.')

In [ ]:
import os, secrets, subprocess, time, requests
os.chdir('/content/Zereos-AI-Shorts')
token=secrets.token_hex(32)
env=os.environ.copy()
env.update({'MUSE_TALK_ROOT':'/content/MuseTalk','MUSE_TALK_WEIGHTS':'/content/MuseTalk/models','AVATAR_WORKER_TOKEN':token,'AVATAR_ALLOWED_ORIGINS':'https://arkanws513-spec.github.io','AVATAR_MAX_UPLOAD_BYTES':str(200*1024*1024)})
worker_log=open('/content/zereos-worker.log','w')
worker=subprocess.Popen(['uvicorn','worker:app','--host','0.0.0.0','--port','7860'],cwd='/content/Zereos-AI-Shorts/avatar-engine',env=env,stdout=worker_log,stderr=subprocess.STDOUT)
time.sleep(8)
try:
    health=requests.get('http://127.0.0.1:7860/api/health',timeout=20).json()
except Exception as e:
    print(open('/content/zereos-worker.log').read()[-5000:])
    raise RuntimeError('Worker did not start; inspect the log above.') from e
print('Worker health:', health)
if not health.get('ready') or not health.get('gpu_available'):
    print('Worker is not ready. Missing weights/runtime:')
    print(health.get('detail'))
    raise RuntimeError('Worker readiness failed. Do not continue until all model files exist.')
print('\nWORKER_TOKEN (copy this into the studio; it is shown only in this session):\n'+token)

In [ ]:
import subprocess, os, time, requests, re
deb='/content/cloudflared-linux-amd64.deb'
subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb','-O',deb],check=True)
subprocess.run(['dpkg','-i',deb],check=True)
tunnel_log=open('/content/zereos-tunnel.log','w')
tunnel=subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(60):
    time.sleep(2)
    tunnel_log.flush()
    log=open('/content/zereos-tunnel.log').read()
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
    if match:
        public_url=match.group(0); break
if not public_url:
    print(open('/content/zereos-tunnel.log').read()[-5000:])
    raise RuntimeError('Cloudflare temporary tunnel did not start.')
print('GPU WORKER URL:',public_url)
print('TOKEN:',token)
print('Open the studio:', 'https://arkanws513-spec.github.io/Zereos-AI-Shorts/real-avatar-studio.html')
print('Enter the URL and token above, click Check connection, then upload a permitted source and audio file.')
print('Keep this notebook session running while generating. The URL/token expire when the session ends.')

## Stop
Stop the Colab runtime when finished. The temporary URL and token are session-only; do not share them publicly. If the health check fails, inspect `/content/zereos-worker.log` and verify all required MuseTalk weights were downloaded.